<a href="https://colab.research.google.com/github/Juanezm/uoc-data-science-tfm/blob/main/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download and concatenate the CSV files:

In [1]:
import pandas as pd
import requests
import os

In [2]:
#from google.colab import data_table # Activar cuando estemos en entorno Colab
#data_table.enable_dataframe_formatter()

In [3]:
# Lista con los identificadores de los sensores
sensor_ids = [
    '200034001951343334363036',
    '270043001951343334363036',
    '380033001951343334363036',
    '46004e000251353337353037',
    '46005a000351353337353037',
    '4e0022000251353337353037',
    '4e0031000251353337353037',
]

In [4]:
# Diccionario con la localización de los sensores Longitud y Latitud
sensor_location = {
    '200034001951343334363036': (40.1138985, -0.0519082),
    '270043001951343334363036': (40.133098, -0.061),
    '380033001951343334363036': (40.20687, 0.015536),
    '46004e000251353337353037': (40.1138985, -0.0519082),
    '46005a000351353337353037': (40.167529, -0.097165),
    '4e0022000251353337353037': (40.1138985, -0.0519082),
    '4e0031000251353337353037': (40.141384, -0.026397)
}

In [5]:
# Lista con información de los sensores
sensors_info = [
    {
        "name": "Temperature",
        "manufacturer": "SparkFun",
        "model": "Si7021",
        "data_interface": "Analog",
        "units": "Centigrade",
        "range": [-10, 85],
        "accuracy": "+/- 0.4 degrees (C)",
        "variable": "air_temperature_raw"
    },
    {
        "name": "Humidity",
        "manufacturer": "SparkFun",
        "model": "Si7021",
        "data_interface": "Analog",
        "units": "Percentage",
        "range": [0, 80],
        "accuracy": "+/- 3 RH",
        "variable": "humidity_raw"
    },
    {
        "name": "Barometric pressure",
        "manufacturer": "SparkFun",
        "model": "MPL3115A2",
        "data_interface": "I2C",
        "units": "Hectopascal",
        "range": [500, 1100],
        "accuracy": "+/- 0.04 hPa",
        "variable": "atmospheric_pressure_raw"
    },
    {
        "name": "Soil moisture",
        "manufacturer": "SparkFun",
        "model": "DS18B20",
        "data_interface": "Analog",
        "units": "Percentage",
        "range": [0, 85],
        "accuracy": "+/- 0.5 RH",
        "variable": "soil_humidity_raw"
    },
    {
        "name": "Wind speed",
        "manufacturer": "SparkFun",
        "model": "SEN08942",
        "data_interface": "Analog (RJ11)",
        "units": "km/h",
        "range": "N/A",
        "accuracy": "N/A",
        "variable": "wind_speed_raw"
    },
    {
        "name": "Wind direction",
        "manufacturer": "SparkFun",
        "model": "SEN08942",
        "data_interface": "Analog (RJ11)",
        "units": "Direction (degrees)",
        "range": [-1, 7],
        "accuracy": "N/A",
        "variable": "wind_direction_raw"
    },
    {
        "name": "Rain meter",
        "manufacturer": "SparkFun",
        "model": "SEN08942",
        "data_interface": "Analog (RJ11)",
        "units": "millilitres (mm)",
        "range": [-1, 7],
        "accuracy": "N/A",
        "variable": "precipitation_raw"
    },
    {
        "name": "Battery",
        "manufacturer": "N/A",
        "model": "N/A",
        "data_interface": "N/A",
        "units": "Percentage",
        "range": [0, 100],
        "accuracy": "N/A",
        "variable": "battery_raw"
    }
]

In [6]:
var_predictoras = ["air_temperature_raw", 
                   "humidity_raw", 
                   "atmospheric_pressure_raw", 
                   "soil_humidity_raw", 
                   "wind_speed_raw", 
                   "wind_direction_raw", 
                   "precipitation_raw", 
                   "battery_raw"]

### El conjunto de datos esta formado por 7 sensores y 8 variables para cada sensor, en total 7x8=56 archivos en formato csv.
### Los nombres de los archivos siguen el formato identificadordelsensor_variable_raw
### El formato de los archivos es csv

In [12]:
# dirección url del repositorio de datos
base_url = "https://zenodo.org/records/3727310/files/"
# directorio para la descarga de los datos brutos
os.makedirs("../src/data_download", exist_ok=True)
download_dir = "../src/data_download"

In [13]:
# Función para descargar un archivo identificado con su id de sensor y la variable a descargar
# Total 56 archivos
def download_csv(sensor_id, variable):
    file_name = f"{sensor_id}_{variable}.csv"
    url = f"{base_url}{file_name}"
    response = requests.get(url)

    with open(f"{download_dir}/{file_name}", "wb") as f:
        f.write(response.content)


In [14]:
# Bucle que recorre los 7 sensores y las 8 variables
for sensor_id in sensor_ids:
    # Bucle que recorre cada una de las variables predictoras
    for var in var_predictoras:
        if not os.path.isfile(f"{download_dir}/{sensor_id}_{var}.csv"): # Verifica que el archivo no existe en el directorio
            download_csv(sensor_id, var) # Descarga el archivo correspondiente

### Ontenemos los datos de duración del día

In [15]:
def get_day_length(latitude, longitude, date):
    """
    Get the length of daylight for a given latitude, longitude, and date.

    Args:
        latitude (float): The latitude of the location.
        longitude (float): The longitude of the location.
        date (str): The date for which to retrieve the daylight length in 'YYYY-MM-DD' format.

    Returns:
        str: The length of daylight in hours and minutes, formatted as 'HH:MM'.

    Example:
        get_day_length(37.7749, -122.4194, '2023-05-29')
        '14:26'
    """
    url = f"https://api.sunrise-sunset.org/json?lat={latitude}&lng={longitude}&date={date}"
    response = requests.get(url)
    sunrise_sunset = response.json()
    return sunrise_sunset.get('results', {}).get('day_length', '')

In [16]:
get_day_length(37.7749, -122.4194, '2024-12-31')

'09:39:05'